#### CCS to icd code conversion

In [ ]:
import pandas as pd
import duckdb
import csv

In [ ]:
target_ccs_codes = ['2', '50', '55', '114', '157', '159', '244']

In [ ]:
# Short names mapping
short_names = {
    '2': 'septicemia',
    '50': 'diabetes',
    '55': 'fluid_electrolyte',
    '114': 'peripheral_athero',
    '157': 'acute_renal_failure',
    '159': 'uti',
    '244': 'other_injury'
}

In [ ]:
# Shared long names used for both ICD-9 and ICD-10
long_names = {
    'septicemia': 'Septicemia',
    'diabetes': 'Diabetes mellitus with complications',
    'fluid_electrolyte': 'Fluid and electrolyte disorders',
    'peripheral_athero': 'Peripheral and visceral atherosclerosis',
    'acute_renal_failure': 'Acute and unspecified renal failure',
    'uti': 'Urinary tract infections',
    'other_injury': 'Other injuries and conditions due to external causes',
}

In [ ]:

# Read both ICD-9 and ICD-10 CSVs
conditions = {}

# Process ICD-9
with open('../data/input/medpar_outcomes_icd_codes/ccs2icd9.csv', 'r', encoding='utf-8') as file:
    reader = csv.reader(file)
    next(reader); next(reader)  # skip note + header
    for row in reader:
        if len(row) >= 2:
            code = row[0].strip().strip("'").replace(' ', '')
            ccs_code = row[1].strip().strip("'").strip()
            if ccs_code in target_ccs_codes:
                ident = short_names[ccs_code]
                conditions.setdefault(ident, {'icd9': [], 'icd10': []})['icd9'].append(code)

# Process ICD-10
with open('../data/input/medpar_outcomes_icd_codes/ccs2icd10.csv', 'r', encoding='utf-8') as file:
    reader = csv.reader(file)
    next(reader)  # skip header
    for row in reader:
        if len(row) >= 2:
            code = row[0].strip().strip("'").replace(' ', '')
            ccs_code = row[1].strip().strip("'").strip()
            if ccs_code in target_ccs_codes:
                ident = short_names[ccs_code]
                conditions.setdefault(ident, {'icd9': [], 'icd10': []})['icd10'].append(code)

# Deduplicate and sort all codes
for ident in conditions:
    conditions[ident]['icd9'] = sorted(set(conditions[ident]['icd9']))
    conditions[ident]['icd10'] = sorted(set(conditions[ident]['icd10']))

In [ ]:
# Generate YAML output
yaml_output = ""
for ident in sorted(conditions.keys()):
    long_name = long_names.get(ident, ident)
    icd9_codes = conditions[ident]['icd9']
    icd10_codes = conditions[ident]['icd10']
    yaml_output += f"\n{ident} :\n"
    yaml_output += f"    long_name : \"{long_name}\"\n"
    yaml_output += f"    icd9 : {icd9_codes}\n"
    yaml_output += f"    icd9_count: {len(icd9_codes)}\n"
    yaml_output += f"    icd10 : {icd10_codes}\n"
    yaml_output += f"    icd10_count: {len(icd10_codes)}\n"

print(yaml_output)

In [ ]:
# Save YAML to file
output_path = '../conf/icd_codes/xiao_raphael_00.yml'

with open(output_path, 'w', encoding='utf-8') as f:
    f.write(yaml_output)

print(f"YAML saved to: {output_path}")